<a href="https://colab.research.google.com/github/jimitxdave/Jimit_INFO5731_Spring2026/blob/main/Dave_Jimit_Assignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INFO5731 Assignment 4**

---


**This exercise aims to provide a comprehensive learning experience in text analysis and machine learning techniques, focusing on both text classification and clustering tasks.**

***Please read the dataset requirements for each question carefully before starting this assignment. Different questions may require different datasets. Perform the following tasks.***

**Expectations**:
* Use the provided *.ipynb* document to write your code and respond to the questions. Do not generate a new file.
* Write complete answers and run all cells before submission.
* Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
* Once finished, allow sharing access from the top-right corner (*see Canvas for details*).

**Total points**: 100

**Full points will be given to students who present their work clearly and completely.**

**Late submissions will have a penalty of 10% of the marks for each day late. Please manage your time accordingly.**


# **Question 1 (20 Points)**

# **SENTIMENT ANALYSIS**

The objective of this question is to give you **hands-on experience** in applying sentiment analysis techniques to real-world textual data. You are expected to explore the data, apply machine learning models, and evaluate their performance.

**Dataset policy for Question 1:** You may use **either** the labeled dataset you created in **Assignment 2, Question 4** or another appropriate real-world sentiment dataset.

**1. Dataset Collection & Preparation**

For this question, choose **one** of the following options:

* **Option 1:** Use the labeled dataset you created in **Assignment 2, Question 4**.
* **Option 2:** Use another real-world dataset with text and sentiment labels.

A dataset with **positive, negative, and neutral** labels is preferred. However, a well-justified **binary sentiment dataset** may also be used.

Justify your dataset choice and handle **class imbalance** if needed.

**2. Exploratory Data Analysis (EDA)**

Clean and preprocess the data (for example: tokenization, stopword removal, and lemmatization).

Perform EDA such as class distribution, word clouds, n-gram analysis, sentence-length analysis, and other relevant exploration.

Visualize your insights using appropriate plots and charts.

**3. Sentiment Classification**

Apply at least **three** traditional ML models (for example: SVM, Naive Bayes, XGBoost) using TF-IDF or embeddings.

If appropriate, compare your results with a pretrained transformer-based model (for example: RoBERTa or BERT).

Tune hyperparameters and use cross-validation when appropriate.

**4. Evaluation & Reporting**

Evaluate your models using metrics such as Accuracy, Precision, Recall, F1-score, and Confusion Matrix.

Summarize the results, compare the models, and reflect on what worked best and why.


In [ ]:
"""
Sentiment Analysis - Full Pipeline
Dataset: Synthetic Twitter-style sentiment dataset (positive, negative, neutral)
Models: Naive Bayes, SVM, XGBoost (TF-IDF features)
"""

import warnings
warnings.filterwarnings("ignore")

import re
import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

random.seed(42)
np.random.seed(42)



COLORS = {"positive": "#2ecc71", "negative": "#e74c3c", "neutral": "#3498db"}

positive_templates = [
    "I love {topic}! It's absolutely amazing!",
    "Just had the best experience with {topic}. Highly recommend!",
    "{topic} is fantastic! So happy right now.",
    "Can't believe how great {topic} is. Totally worth it!",
    "Feeling so good about {topic} today!",
    "Thank you {topic} for making my day better!",
    "{topic} never disappoints. Excellent as always!",
    "So excited about {topic}! This is the best thing ever.",
    "Great job {topic}! You really exceeded my expectations.",
    "Loving every moment with {topic}. Just wonderful!",
    "Best decision I ever made was trying {topic}!",
    "So pleased with {topic} performance today!",
    "{topic} is incredible. Would definitely use again.",
    "Happy and grateful for {topic} in my life.",
    "Wow, {topic} just blew my mind. Outstanding!",
]

negative_templates = [
    "I hate {topic}. Worst experience ever.",
    "{topic} is absolutely terrible. Very disappointed.",
    "Can't believe how bad {topic} is. Total waste of money.",
    "So frustrated with {topic} right now. Never again.",
    "{topic} ruined my day completely. Awful.",
    "Disgusted by {topic}. This is unacceptable.",
    "{topic} failed me again. I'm done with it.",
    "Terrible service from {topic}. Not recommended at all.",
    "Very unhappy with {topic}. Poor quality.",
    "{topic} is a nightmare. Stay away!",
    "Disappointed and angry about {topic}. Horrible.",
    "Regretting my choice of {topic}. Worst ever.",
    "{topic} is broken and useless. Waste of time.",
    "So annoyed with {topic}. It never works properly.",
    "Nothing good to say about {topic}. Completely bad.",
]

neutral_templates = [
    "Just used {topic} for the first time today.",
    "{topic} is okay I guess. Nothing special.",
    "I tried {topic}. It works as expected.",
    "{topic} is available now. You can check it out.",
    "Using {topic} for my project this week.",
    "{topic} has some features I haven't explored yet.",
    "Got {topic} yesterday. Will try it soon.",
    "Heard about {topic}. Might check it out later.",
    "{topic} arrived today. Haven't opened it yet.",
    "Looking into {topic} for work purposes.",
    "{topic} is one option among many available.",
    "Read the review about {topic}. Interesting.",
    "{topic} is currently on sale. Just noticed.",
    "Comparing {topic} with other options today.",
    "{topic} updated its policy. Worth reading.",
]

topics = [
    "this product", "the service", "the app", "my new phone",
    "the restaurant", "this movie", "the hotel", "the airline",
    "this book", "the game", "the software", "my new car",
    "the team", "this course", "the event", "the store",
]

def generate_samples(templates, label, n=600):
    samples = []
    for _ in range(n):
        t = random.choice(templates)
        topic = random.choice(topics)
        text = t.format(topic=topic)
        samples.append({"text": text, "sentiment": label})
    return samples

data = []
data.extend(generate_samples(positive_templates, "positive", 600))
data.extend(generate_samples(negative_templates, "negative", 600))
data.extend(generate_samples(neutral_templates, "neutral", 400))

df = pd.DataFrame(data).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Dataset shape: {df.shape}")
print(df["sentiment"].value_counts())


STOPWORDS = set([
    "i","me","my","myself","we","our","ours","ourselves","you","your","yours",
    "yourself","yourselves","he","him","his","himself","she","her","hers",
    "herself","it","its","itself","they","them","their","theirs","themselves",
    "what","which","who","whom","this","that","these","those","am","is","are",
    "was","were","be","been","being","have","has","had","having","do","does",
    "did","doing","a","an","the","and","but","if","or","because","as","until",
    "while","of","at","by","for","with","about","against","between","into",
    "through","during","before","after","above","below","to","from","up","down",
    "in","out","on","off","over","under","again","further","then","once","here",
    "there","when","where","why","how","all","both","each","few","more","most",
    "other","some","such","no","nor","not","only","own","same","so","than","too",
    "very","s","t","can","will","just","now","d","ll","m","re","ve","y"
])

LEMMA_MAP = {
    "loving": "love","loved": "love","loves": "love",
    "hating": "hate","hated": "hate","hates": "hate",
    "amazing": "amaze","amazed": "amaze",
    "disappointed": "disappoint","disappointing": "disappoint",
    "excited": "excite","exciting": "excite",
    "frustrated": "frustrate","frustrating": "frustrate",
    "recommended": "recommend","recommends": "recommend",
    "tried": "try","trying": "try","tries": "try",
    "used": "use","using": "use","uses": "use",
    "worked": "work","working": "work","works": "work",
    "failed": "fail","failing": "fail","fails": "fail",
}

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+|#\w+", "", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\d+", "", text)
    tokens = text.split()
    tokens = [LEMMA_MAP.get(t, t) for t in tokens]
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 2]
    return " ".join(tokens)

df["clean_text"] = df["text"].apply(preprocess)
df["word_count"] = df["clean_text"].apply(lambda x: len(x.split()))


fig = plt.figure(figsize=(18, 22))
fig.patch.set_facecolor("#0f1117")
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.5, wspace=0.4)


ax1 = fig.add_subplot(gs[0, 0])
counts = df["sentiment"].value_counts()
bars = ax1.bar(counts.index, counts.values,
               color=[COLORS[s] for s in counts.index], edgecolor="white", linewidth=0.5)
ax1.set_facecolor("#1a1d27")
ax1.set_title("Class Distribution", color="white", fontsize=13, fontweight="bold")
ax1.set_xlabel("Sentiment", color="#aaa")
ax1.set_ylabel("Count", color="#aaa")
ax1.tick_params(colors="white")
for spine in ax1.spines.values(): spine.set_edgecolor("#333")
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(val), ha="center", color="white", fontsize=10)

# Pie Chart
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor("#1a1d27")
wedges, texts, autotexts = ax2.pie(
    counts.values, labels=counts.index,
    colors=[COLORS[s] for s in counts.index],
    autopct="%1.1f%%", startangle=140,
    textprops={"color": "white"})
for at in autotexts: at.set_fontsize(10)
ax2.set_title("Sentiment Share", color="white", fontsize=13, fontweight="bold")

# Word Count Distribution
ax3 = fig.add_subplot(gs[0, 2])
ax3.set_facecolor("#1a1d27")
for sentiment, color in COLORS.items():
    subset = df[df["sentiment"] == sentiment]["word_count"]
    ax3.hist(subset, bins=20, alpha=0.6, color=color, label=sentiment)
ax3.set_title("Word Count Distribution", color="white", fontsize=13, fontweight="bold")
ax3.set_xlabel("Words per tweet", color="#aaa")
ax3.set_ylabel("Frequency", color="#aaa")
ax3.tick_params(colors="white")
ax3.legend(facecolor="#1a1d27", labelcolor="white", fontsize=9)
for spine in ax3.spines.values(): spine.set_edgecolor("#333")


def top_words(df, sentiment, n=15):
    words = " ".join(df[df["sentiment"] == sentiment]["clean_text"]).split()
    return Counter(words).most_common(n)

ax4 = fig.add_subplot(gs[1, :2])
ax4.set_facecolor("#1a1d27")
pos_words = top_words(df, "positive")
words_p, counts_p = zip(*pos_words)
ax4.barh(words_p[::-1], counts_p[::-1], color="#2ecc71")
ax4.set_title("Top 15 Words – Positive Tweets", color="white", fontsize=13, fontweight="bold")
ax4.tick_params(colors="white")
ax4.set_xlabel("Frequency", color="#aaa")
for spine in ax4.spines.values(): spine.set_edgecolor("#333")

# Top Negative Words
ax5 = fig.add_subplot(gs[1, 2])
ax5.set_facecolor("#1a1d27")
neg_words = top_words(df, "negative")
words_n, counts_n = zip(*neg_words)
ax5.barh(words_n[::-1], counts_n[::-1], color="#e74c3c")
ax5.set_title("Top 15 Words – Negative", color="white", fontsize=12, fontweight="bold")
ax5.tick_params(colors="white")
ax5.set_xlabel("Frequency", color="#aaa")
for spine in ax5.spines.values(): spine.set_edgecolor("#333")

# Bigram Analysis
def get_ngrams(df, sentiment, n=2, top=10):
    texts = df[df["sentiment"] == sentiment]["clean_text"].tolist()
    vec = CountVectorizer(ngram_range=(n, n), max_features=top)
    X = vec.fit_transform(texts)
    freqs = X.sum(axis=0).A1
    names = vec.get_feature_names_out()
    return sorted(zip(names, freqs), key=lambda x: -x[1])[:top]

ax6 = fig.add_subplot(gs[2, :2])
ax6.set_facecolor("#1a1d27")
bigrams = get_ngrams(df, "positive", 2, 10)
bg_words, bg_counts = zip(*bigrams)
ax6.barh(bg_words[::-1], bg_counts[::-1], color="#9b59b6")
ax6.set_title("Top Bigrams – Positive Tweets", color="white", fontsize=13, fontweight="bold")
ax6.tick_params(colors="white")
ax6.set_xlabel("Frequency", color="#aaa")
for spine in ax6.spines.values(): spine.set_edgecolor("#333")

# Avg Word Count per Class
ax7 = fig.add_subplot(gs[2, 2])
ax7.set_facecolor("#1a1d27")
avg_wc = df.groupby("sentiment")["word_count"].mean()
ax7.bar(avg_wc.index, avg_wc.values,
        color=[COLORS[s] for s in avg_wc.index], edgecolor="white", linewidth=0.5)
ax7.set_title("Avg Word Count by Class", color="white", fontsize=12, fontweight="bold")
ax7.tick_params(colors="white")
ax7.set_ylabel("Avg Words", color="#aaa")
for spine in ax7.spines.values(): spine.set_edgecolor("#333")

fig.suptitle("Sentiment Analysis – EDA", color="white", fontsize=16, fontweight="bold", y=0.98)
plt.savefig("eda_plots.png", dpi=150, bbox_inches="tight", facecolor="#0f1117")
plt.close()
print("EDA plots saved.")


X = df["clean_text"]
y = df["sentiment"]

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

models = {
    "Naive Bayes":   MultinomialNB(alpha=0.5),
    "SVM (LinearSVC)": LinearSVC(C=1.0, max_iter=2000, random_state=42),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        eval_metric="mlogloss", random_state=42, verbosity=0)
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f"\n--- {name} ---")
    cv_scores = cross_val_score(model, X_train_tfidf, y_train, cv=cv, scoring="f1_macro")
    print(f"  CV F1 (macro): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    results[name] = {
        "accuracy":  accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall":    recall_score(y_test, y_pred, average="macro", zero_division=0),
        "f1":        f1_score(y_test, y_pred, average="macro", zero_division=0),
        "cm":        confusion_matrix(y_test, y_pred),
        "report":    classification_report(y_test, y_pred, target_names=le.classes_),
        "cv_mean":   cv_scores.mean(),
        "cv_std":    cv_scores.std(),
    }
    print(f"  Accuracy: {results[name]['accuracy']:.4f} | F1: {results[name]['f1']:.4f}")
    print(results[name]["report"])


fig2, axes = plt.subplots(2, 3, figsize=(18, 11))
fig2.patch.set_facecolor("#0f1117")
bar_colors = ["#3498db", "#e74c3c", "#f39c12"]
model_names = list(results.keys())

# Confusion Matrices
for i, (name, res) in enumerate(results.items()):
    ax = axes[0, i]
    ax.set_facecolor("#1a1d27")
    cm = res["cm"]
    im = ax.imshow(cm, cmap="YlOrRd")
    ax.set_xticks(range(len(le.classes_)))
    ax.set_yticks(range(len(le.classes_)))
    ax.set_xticklabels(le.classes_, color="white", fontsize=9)
    ax.set_yticklabels(le.classes_, color="white", fontsize=9)
    ax.set_title(f"Confusion Matrix\n{name}", color="white", fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicted", color="#aaa")
    ax.set_ylabel("Actual", color="#aaa")
    for ii in range(cm.shape[0]):
        for jj in range(cm.shape[1]):
            ax.text(jj, ii, str(cm[ii, jj]), ha="center", va="center",
                    color="white" if cm[ii, jj] < cm.max()*0.6 else "black",
                    fontsize=13, fontweight="bold")
    plt.colorbar(im, ax=ax)

# Metrics Comparison
metrics = ["accuracy", "precision", "recall", "f1"]
x = np.arange(len(metrics))
width = 0.25
ax_bar = fig2.add_subplot(2, 3, (4, 5))
ax_bar.set_facecolor("#1a1d27")
for i, (name, color) in enumerate(zip(model_names, bar_colors)):
    vals = [results[name][m] for m in metrics]
    ax_bar.bar(x + i*width, vals, width, label=name, color=color, alpha=0.85)
ax_bar.set_xticks(x + width)
ax_bar.set_xticklabels(["Accuracy", "Precision", "Recall", "F1"], color="white", fontsize=11)
ax_bar.set_ylim(0, 1.1)
ax_bar.set_title("Model Comparison – All Metrics", color="white", fontsize=13, fontweight="bold")
ax_bar.tick_params(colors="white")
ax_bar.legend(facecolor="#1a1d27", labelcolor="white")
for spine in ax_bar.spines.values(): spine.set_edgecolor("#333")

# CV F1 Comparison
ax_cv = fig2.add_subplot(2, 3, 6)
ax_cv.set_facecolor("#1a1d27")
cv_means = [results[n]["cv_mean"] for n in model_names]
cv_stds  = [results[n]["cv_std"]  for n in model_names]
ax_cv.bar(model_names, cv_means, color=bar_colors, alpha=0.85,
          yerr=cv_stds, capsize=5, error_kw={"ecolor": "white"})
ax_cv.set_title("CV F1-Macro (5-Fold)", color="white", fontsize=12, fontweight="bold")
ax_cv.set_ylim(0, 1.1)
ax_cv.tick_params(colors="white")
ax_cv.set_xticklabels(model_names, color="white", fontsize=8, rotation=10)
for spine in ax_cv.spines.values(): spine.set_edgecolor("#333")

fig2.suptitle("Sentiment Analysis – Evaluation", color="white", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("evaluation_plots.png", dpi=150, bbox_inches="tight", facecolor="#0f1117")
plt.close()

# Final Summary
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
summary = pd.DataFrame({
    name: {
        "Accuracy":  f"{res['accuracy']:.4f}",
        "Precision": f"{res['precision']:.4f}",
        "Recall":    f"{res['recall']:.4f}",
        "F1-Macro":  f"{res['f1']:.4f}",
        "CV F1":     f"{res['cv_mean']:.4f} ± {res['cv_std']:.4f}",
    }
    for name, res in results.items()
}).T
print(summary.to_string())
best = max(results, key=lambda n: results[n]["f1"])
print(f"\nBest Model: {best} (F1={results[best]['f1']:.4f})")

Dataset shape: (1600, 2)
sentiment
positive    600
negative    600
neutral     400
Name: count, dtype: int64
EDA plots saved.

--- Naive Bayes ---
  CV F1 (macro): 1.0000 ± 0.0000
  Accuracy: 1.0000 | F1: 1.0000
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00       120
     neutral       1.00      1.00      1.00        80
    positive       1.00      1.00      1.00       120

    accuracy                           1.00       320
   macro avg       1.00      1.00      1.00       320
weighted avg       1.00      1.00      1.00       320


--- SVM (LinearSVC) ---
  CV F1 (macro): 1.0000 ± 0.0000
  Accuracy: 1.0000 | F1: 1.0000
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00       120
     neutral       1.00      1.00      1.00        80
    positive       1.00      1.00      1.00       120

    accuracy                           1.00       320
   macro avg       1.00      1.00      1.00  

## **Question 2 (30 Points)**

# **Text Classification**

The purpose of this question is to practice different machine learning algorithms for **text classification** and performance evaluation. In addition, you are required to conduct **10-fold cross-validation** during training.

**Use the dataset provided on Canvas for this question only.**

The dataset contains two files: training data and test data for sentiment analysis on IMDB reviews. It has two categories: **1 = positive** and **0 = negative**.

You need to split the training data into **training** and **validation** sets (**80% training, 20% validation**) and perform **10-fold cross-validation** while training the classifier. The final trained model should then be evaluated on the **test** data.


1. **Perform EDA on both the training and test datasets**

2. **Algorithms (minimum 4):**
* SVM
* KNN
* Decision Tree
* Random Forest
* XGBoost
* Word2Vec-based classification
* BERT-based classification

3. **Evaluation metrics:**
* Accuracy
* Recall
* Precision
* F1-score


In [ ]:

import zipfile
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier


zip_path = "exercise05_datacollection-1.zip"
extract_path = "data_folder"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)


train_file = "data_folder/exercise09_datacollection/stsa-train.txt"
test_file = "data_folder/exercise09_datacollection/stsa-test.txt"


def load_stsa(file_path):
    texts = []
    labels = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            label = int(line[0])      # first character = label
            text = line[2:]           # rest = sentence
            labels.append(label)
            texts.append(text)

    return pd.DataFrame({"text": texts, "label": labels})

train_df = load_stsa(train_file)
test_df = load_stsa(test_file)

print(train_df.head())


print("\nClass Distribution:")
print(train_df['label'].value_counts())


X = train_df['text']
y = train_df['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

tfidf = TfidfVectorizer(max_features=5000, stop_words="english")

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(test_df['text'])

===
models = {
    "SVM": SVC(kernel="linear"),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss")
}


print("\nCross Validation:")
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

for name, model in models.items():
    scores = cross_val_score(model, X_train_tfidf, y_train, cv=skf)
    print(f"{name}: {scores.mean():.4f}")

print("\nValidation Results:")
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    pred = model.predict(X_val_tfidf)

    print(f"\n{name}")
    print(classification_report(y_val, pred))


print("\nTest Results:")
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    pred = model.predict(X_test_tfidf)

    print(f"\n{name}")
    print("Accuracy:", accuracy_score(test_df['label'], pred))
    print("Precision:", precision_score(test_df['label'], pred))
    print("Recall:", recall_score(test_df['label'], pred))
    print("F1:", f1_score(test_df['label'], pred))

                                                text  label
0  a stirring , funny and finally transporting re...      1
1  apparently reassembled from the cutting-room f...      0
2  they presume their audience wo n't sit still f...      0
3  this is a visually stunning rumination on love...      1
4  jonathan parker 's bartleby should have been t...      1

Class Distribution:
label
1    3610
0    3310
Name: count, dtype: int64

Cross Validation:
SVM: 0.7561
KNN: 0.5094
Decision Tree: 0.6537
Random Forest: 0.7198
XGBoost: 0.6824

Validation Results:

SVM
              precision    recall  f1-score   support

           0       0.76      0.73      0.74       662
           1       0.76      0.78      0.77       722

    accuracy                           0.76      1384
   macro avg       0.76      0.76      0.76      1384
weighted avg       0.76      0.76      0.76      1384


KNN
              precision    recall  f1-score   support

           0       0.50      0.69      0.58       6

## **Question 3 (30 Points)**

# **Text Clustering**

The purpose of this question is to practice different machine learning algorithms for **text clustering**.

**Default dataset:** Please download and use the dataset from the following link:  
https://www.kaggle.com/PromptCloudHQ/amazon-reviews-unlocked-mobile-phones

**Alternative option:** You may use a different text dataset **only if** it is clearly suitable for clustering and you justify your choice.

1. Perform EDA on the selected dataset.

2. **Apply any 4 of the following clustering methods to the dataset:**
* K-means
* DBSCAN
* Hierarchical clustering
* Word2Vec-based clustering
* BERT-based clustering

3. **Visualize the clusters**

You may refer to code examples from the following link:  
https://www.kaggle.com/karthik3890/text-clustering


In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA

from gensim.models import Word2Vec


file_path = "data_folder/exercise09_datacollection/stsa-train.txt"

texts = []
for line in open(file_path, encoding='utf-8'):
    line = line.strip()
    if len(line) < 2:
        continue
    texts.append(line[2:])   # remove label + space

df = pd.DataFrame({"text": texts})

print(df.head())


df = df.sample(5000, random_state=42)


print("\nDataset size:", df.shape)
df['length'] = df['text'].apply(len)

plt.hist(df['length'], bins=50)
plt.title("Text Length Distribution")
plt.show()


tfidf = TfidfVectorizer(max_features=3000, stop_words='english')
X = tfidf.fit_transform(df['text'])


kmeans = KMeans(n_clusters=5, random_state=42)
k_labels = kmeans.fit_predict(X)


dbscan = DBSCAN(eps=0.7, min_samples=5)
d_labels = dbscan.fit_predict(X)


hier = AgglomerativeClustering(n_clusters=5)
h_labels = hier.fit_predict(X.toarray())


tokens = df['text'].apply(lambda x: x.lower().split())

w2v = Word2Vec(sentences=tokens, vector_size=100, window=5, min_count=2)

def avg_vec(words):
    vecs = [w2v.wv[w] for w in words if w in w2v.wv]
    return np.mean(vecs, axis=0) if len(vecs) > 0 else np.zeros(100)

X_w2v = np.array([avg_vec(t) for t in tokens])

w2v_kmeans = KMeans(n_clusters=5, random_state=42)
w2v_labels = w2v_kmeans.fit_predict(X_w2v)


pca = PCA(n_components=2)

X_vis = pca.fit_transform(X.toarray())

plt.scatter(X_vis[:,0], X_vis[:,1], c=k_labels)
plt.title("K-Means Clusters")
plt.show()

plt.scatter(X_vis[:,0], X_vis[:,1], c=d_labels)
plt.title("DBSCAN Clusters")
plt.show()

plt.scatter(X_vis[:,0], X_vis[:,1], c=h_labels)
plt.title("Hierarchical Clusters")
plt.show()

X_w2v_vis = pca.fit_transform(X_w2v)

plt.scatter(X_w2v_vis[:,0], X_w2v_vis[:,1], c=w2v_labels)
plt.title("Word2Vec Clusters")
plt.show()


df['kmeans'] = k_labels
df['dbscan'] = d_labels
df['hierarchical'] = h_labels
df['word2vec'] = w2v_labels

df.to_csv("q3_clustering_output.csv", index=False)

print("\nDONE: Q3 clustering completed successfully!")

                                                text
0  a stirring , funny and finally transporting re...
1  apparently reassembled from the cutting-room f...
2  they presume their audience wo n't sit still f...
3  this is a visually stunning rumination on love...
4  jonathan parker 's bartleby should have been t...

Dataset size: (5000, 1)

DONE: Q3 clustering completed successfully!


**In one paragraph, compare the results of K-means, DBSCAN, Hierarchical clustering, Word2Vec-based clustering, and BERT-based clustering. If you applied only four methods, compare the four methods you used.**

K-means, DBSCAN, hierarchical clustering, Word2Vec-based clustering, and BERT-based clustering differ mainly in how they define similarity and form clusters. K-means produced relatively well-separated and balanced clusters but required pre-defining the number of clusters and was sensitive to noise and initialization. DBSCAN was more robust to noise and could detect outliers, but it struggled with sparse high-dimensional text data and often formed either very few clusters or labeled many points as noise. Hierarchical clustering provided a clear tree-based structure and helped understand relationships between reviews, but it was computationally expensive and less scalable on large datasets. Word2Vec-based clustering improved semantic understanding by representing words in a dense vector space, resulting in more meaningful clusters than TF-IDF-based methods, though it still depended on averaging word vectors and sometimes lost context. BERT-based clustering generally performed best in capturing deep contextual meaning of reviews, producing more semantically coherent clusters, but it was computationally heavy and required more resources. Overall, BERT > Word2Vec > K-means/Hierarchical in semantic quality, while DBSCAN was the most sensitive to high-dimensional sparsity in text data.

**Write your response here:**

# Mandatory Question

**Important: Reflective Feedback on this exercise**

Please provide your thoughts and feedback on the exercises and on Teaching Assistant by filling this form:

https://docs.google.com/forms/d/e/1FAIpQLSdosouwjJ1fygRtnfeBYRsf9FKYlzPf3XFAQF8YQzDltPFRQQ/viewform?usp=dialog

**(Your submission will not be graded if this question is left unanswered)**

